# Hugging Face 모델 탐색 실습

이번 실습에서는 Hugging Face에서 제공하는 다양한 인공지능 모델을 직접 찾아보고, Colab에서 실행해보겠습니다.

오늘은 모델을 직접 학습시키는 것이 아니라, 이미 학습되어 공개된 모델을 불러와 사용하는 방법을 연습합니다.

이번 시간에 실습할 task는 다음 4가지입니다.

1. 감정분석
2. 번역
3. 한국어 요약
4. 문장 유사도 비교


#라이브러리 설치

Hugging Face 모델을 사용하기 위해 필요한 라이브러리를 먼저 설치하겠습니다.

아래 코드는 실습에 필요한 기본 라이브러리를 설치하는 코드입니다.
처음 한 번만 실행하면 됩니다.

transformers: Hugging Face 모델을 불러오고 실행하는 라이브러리입니다.
sentencepiece: 일부 번역 및 요약 모델에서 사용하는 토크나이저 라이브러리입니다.
sentence-transformers: 문장을 벡터로 바꾸고 유사도를 계산할 때 사용하는 라이브러리입니다.

In [29]:
!pip install -q transformers sentencepiece sentence-transformers accelerate

In [30]:
import transformers
import sentence_transformers

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

transformers: 5.12.1
sentence-transformers: 5.6.0


#Hugging Face 모델 검색 방법

Hugging Face에는 다양한 인공지능 모델이 공개되어 있습니다.

이번 실습에서는 원하는 task에 맞는 모델을 직접 검색해보고, Colab에서 실행해보겠습니다.

모델을 찾는 순서는 다음과 같습니다.

1. Hugging Face 사이트에 접속합니다.
2. 상단 메뉴에서 `Models`를 클릭합니다.
3. 왼쪽 메뉴에서 원하는 `Task`를 선택합니다.
4. 검색창에 원하는 키워드를 입력합니다.
5. 마음에 드는 모델 페이지에 들어갑니다.
6. 모델 이름을 복사합니다.
7. Colab 코드의 `model_name` 부분에 붙여넣습니다.

모델 이름은 보통 다음과 같은 형태입니다.

```text
사용자명/모델명
```

예를 들어 다음과 같은 이름을 사용할 수 있습니다.

```text
distilbert-base-uncased-finetuned-sst-2-english
eenzeenee/t5-base-korean-summarization
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

처음에는 어떤 모델을 골라야 할지 헷갈릴 수 있습니다.
그럴 때는 다운로드 수가 많거나, 모델 카드에 사용 예시가 잘 적혀 있는 모델을 먼저 선택해보면 좋습니다.


#1. 감정분석
- sentiment-analysis
- text-classificatio

In [18]:
from transformers import pipeline
pipe = pipeline("sentiment-analysis", model="rishigupta04/yt-comments-sentiment-analyzer")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [44]:
pipe.model.config.id2label

{0: 'negative', 1: 'neutral', 2: 'positive'}

유튜브 댓글 반응 분석

테스트

In [19]:
pipe("This video was really helpful. Thank you!")

[{'label': 'positive', 'score': 0.9971994161605835}]

In [20]:
pipe("This is the worst tutorial I have ever seen.")

[{'label': 'negative', 'score': 0.9925049543380737}]

In [21]:
pipe("The video explains the topic.")

[{'label': 'neutral', 'score': 0.9211329817771912}]

#2. 번역
- translation

transfomers 5.12.1 버전에서는 translation 작업이 보이지 않아 seq2seq 모델을 직접 불러와서 작업

In [34]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [35]:
def translate(text, target_lang="kor_Hang", max_length=128):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang),
            max_length=max_length
        )

    return tokenizer.batch_decode(
        output_tokens,
        skip_special_tokens=True
    )[0]

테스트

In [36]:
text = "This game is really fun and addictive."
print(translate(text, target_lang="kor_Hang"))

이 게임은 정말 재미있고 중독성이 있습니다.


In [37]:
text = "이 게임은 정말 재미있고 중독성이 있습니다."
print(translate(text, target_lang="eng_Latn"))

This game is really fun and addictive.


#3. 요약
- summarization
- text2text-generation

파이프라인 task 미사용

In [ ]:
summarization = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

In [33]:
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

WHITESPACE_HANDLER = lambda k: re.sub('\s+', ' ', re.sub('\n+', ' ', k.strip()))

article_text = """푸른 달이 떠오른 밤, 초보자 검사 하린은 오래된 온라인 게임 속 왕국에 접속했다. 마을 광장에는 파티를 찾는 모험가들이 모였고, 대장장이는 반짝이는 장비를 두드렸다. 하린은 친구들과 음성 채팅을 켜고 버려진 성채로 향했다. 첫 방에서는 슬라임이 튀어 올랐고, 두 번째 방에서는 독 안개가 길을 막았다. 모두가 역할을 나누어 방패로 공격을 받아내고, 마법으로 적을 묶고, 활로 약점을 노렸다. 마지막 문이 열리자 거대한 기계 용이 나타나 붉은 불꽃을 뿜었다. 체력이 거의 남지 않았지만 하린은 회피 타이밍을 맞춰 필살기를 성공시켰고, 화면에는 승리라는 글자가 떠올랐다. 보상 상자에서 전설 등급 검이 빛났고, 친구들은 환호하며 다음 던전을 약속했다.그날 이후 하린은 단순히 레벨을 올리는 것보다 함께 전략을 세우고 실패를 웃으며 다시 도전하는 순간이 게임의 진짜 재미라는 것을 깨달았다. 그래서 매일 저녁 접속 알림이 울리면, 그는 현실의 피로를 잠시 내려놓고 새로운 모험을 향해 미소 지었다."""

model_name = "csebuetnlp/mT5_multilingual_XLSum"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

input_ids = tokenizer(
    [WHITESPACE_HANDLER(article_text)],
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=512
)["input_ids"]

output_ids = model.generate(
    input_ids=input_ids,
    max_length=84,
    no_repeat_ngram_size=2,
    num_beams=4
)[0]

summary = tokenizer.decode(
    output_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

print(summary)


<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1688/554400420.py:4: SyntaxWarning: invalid escape sequence '\s'
  WHITESPACE_HANDLER = lambda k: re.sub('\s+', ' ', re.sub('\n+', ' ', k.strip()))


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


온라인 게임 속 왕국 접속 알림이 울리면, 게임의 재미는 어떨까?


#4. 문장 유사도 비교
- sentence-similarity
- sentence-transformers 라이브러리 사용

In [48]:
pip install -U sentence-transformers

In [50]:
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
model = SentenceTransformer("upskyy/bge-m3-korean")

# Run inference
sentences = [
    '아이를 가진 엄마가 해변을 걷는다.',
    '두 사람이 해변을 걷는다.',
    '한 남자가 해변에서 개를 산책시킨다.',
]
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 1024]

# Get the similarity scores for the embeddings
similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)
# [3, 3]
print(similarities)
# tensor([[1.0000, 0.6173, 0.3672],
#         [0.6173, 1.0000, 0.4775],
#         [0.3672, 0.4775, 1.0000]])


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

(3, 1024)
torch.Size([3, 3])
tensor([[1.0000, 0.6173, 0.3672],
        [0.6173, 1.0000, 0.4775],
        [0.3672, 0.4775, 1.0000]])


transfomers 사용

In [51]:
from transformers import AutoTokenizer, AutoModel
import torch


# Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


# Sentences we want sentence embeddings for
sentences = ["안녕하세요?", "한국어 문장 임베딩을 위한 버트 모델입니다."]

# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained("upskyy/bge-m3-korean")
model = AutoModel.from_pretrained("upskyy/bge-m3-korean")

# Tokenize sentences
encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling. In this case, mean pooling.
sentence_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])

print("Sentence embeddings:")
print(sentence_embeddings)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Sentence embeddings:
tensor([[ 0.5602, -0.2838, -0.3859,  ...,  1.2832, -0.2737,  0.6704],
        [ 0.3032,  0.2706,  1.2415,  ...,  0.5054, -1.0032, -1.3843]])


유사도 확인